# Set Up Environment (Local Version)

This is a local-Jupyter version of the Colab notebook.

**One-time setup**, done already if you used VS Code's "Quick Create venv" environment manager (it creates `.venv` in the repo root and installs `requirements.txt` for you). If you ever need to do it by hand instead, from a terminal in the repo root:
```powershell
py -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
```
Either way, open this notebook in VS Code, click the kernel picker in the top right, and select the **.venv** interpreter for this workspace — no manual `ipykernel install` step needed, VS Code handles that automatically once `.venv` is selected.


In [ ]:
# Sanity check: confirm the notebook is using the venv's Python, and that we're in the repo root
import sys, os
print('Python executable:', sys.executable)
print('Working directory:', os.getcwd())
# If the working directory isn't the repo root, uncomment and adjust:
# %cd /path/to/hamiltonian-nn


# Creating Dataset and Training Models


## Dataset


In [ ]:
# Creates dataset for model training and visualization
# (uses the parallelized sample_orbits -- pass num_workers=N to data3dcanonical.make_orbits_dataset
#  if you ever want to call it directly from a cell instead of via the script)
!{sys.executable} experiment-3body/3D/make_3d_dataset.py


python experiment-3body/3D/make_3d_dataset.py # Type this in powershell for project bar

In [ ]:
# Displays the dataset
!{sys.executable} experiment-3body/3D/show_3d_dataset.py


## Training


In [ ]:
!{sys.executable} -u experiment-3body/3D/train3DtestcodeTesting.py --verbose



# Visualizations


In [ ]:
# runs script for visualization
# %matplotlib widget

# If ipympl/widget still misbehaves locally, fall back to:
# %matplotlib inline

# For rotating visuals that open in a separate window, use:
import torch
%matplotlib qt


%run experiment-3body/3D/visuals3dcanonical.py

# Restart the kernel if the line above doesn't work, then re-run the sanity-check cell first


## Ground truth


In [ ]:
# displays orbits from dataset
plot_ground_truth(plot3d=True)


## Baseline Model


In [ ]:
# Loads baseline and hnn models (models need to be trained first before visualizations related to models can be displayed)

def load_model(args, baseline=False):
    output_dim = args.input_dim if baseline else 1
    nn_model = MLP(args.input_dim, args.hidden_dim, output_dim, args.nonlinearity)
    model = HNN(args.input_dim, differentiable_model=nn_model,
            field_type=args.field_type, baseline=baseline)
    case = 'baseline' if baseline else 'hnn'
    path = "{}/{}-orbits-{}.tar".format(args.save_dir, args.name, case)
    print(path)
    model.load_state_dict(torch.load(path))
    return model

args = ObjectView(get_args())
base_model = load_model(args, baseline=True)
hnn_model = load_model(args, baseline=False)


In [ ]:
what_has_baseline_learned(base_model, args)


## HNN Model


In [ ]:
what_has_hnn_learned(hnn_model, args)


## All orbits


In [ ]:
# Need to run 'what_has_..._learned()' cells firs
visualize_all_orbits(base_model, hnn_model, args)


In [ ]:
# for debugging...

import pickle, numpy as np
with open('experiment-3body/3D/3body-3Dorbits-dataset.pkl', 'rb') as f:
    d = pickle.load(f)
for k in ['coords', 'dcoords', 'energy']:
    a = d[k]
    print(k, a.shape, 'std', a.std(), 'min', a.min(), 'max', a.max())
dc = np.abs(np.concatenate([d['test_dcoords'], d['dcoords']]))
print('dcoords percentiles (50, 99, 99.9, 100):', np.percentile(dc, [50, 99, 99.9, 100]))
per_orbit = dc.reshape(-1, 20, 18).max(axis=(1, 2))
print('orbits:', len(per_orbit), '| worst-value percentiles (50, 90, 99, 100):', np.percentile(per_orbit, [50, 90, 99, 100]))
for t in [10, 50, 100, 1000]:
    print('orbits with worst |dcoord| >', t, ':', (per_orbit > t).sum())
print('energy percentiles (0, 1, 50, 99, 100):', np.percentile(d['energy'], [0, 1, 50, 99, 100]))
print({k: v for k, v in d['meta'].items() if k in ['timesteps', 'trials', 'orbit_noise', 'min_radius', 'max_radius', 't_span']})

## Energies


In [ ]:
# Need to run 'what_has_..._learned()' cells first
visualize_all_energies(base_model, hnn_model, args)


## Energy Conservation


In [ ]:
# Need to run 'what_has_..._learned()' cells firs
visualize_energy_conservation(base_model, hnn_model, args)


## Training Curves


In [ ]:
# Need to run 'what_has_..._learned()' cells firs
plot_training_curves()


Testing

In [ ]:
import sys, torch, numpy as np
sys.path.append('experiment-3body/3D')
from data3dcanonical import sample_orbits

# Hard test set: same generator, but with the extreme-orbit filter turned off
hard, _ = sample_orbits(trials=300, max_derivative=None, num_workers=1)

x = torch.tensor(hard['coords'], dtype=torch.float32, requires_grad=True)
dxdt = torch.tensor(hard['dcoords'], dtype=torch.float32)
extreme = np.abs(hard['dcoords']).max(axis=1) > 20   # samples the training filter would have removed

for name, model in [('baseline', base_model), ('hnn', hnn_model)]:
    err = ((model.time_derivative(x) - dxdt) ** 2).mean(dim=1).detach().numpy()
    print(f"{name:9s} median error {np.median(err):.3g} | normal samples {err[~extreme].mean():.3g} | extreme samples {err[extreme].mean():.3g}")

more test

In [ ]:
import pickle, numpy as np
with open('experiment-3body/3D/3body-3Dorbits-dataset.pkl', 'rb') as f:
    d = pickle.load(f)
print('max |dcoords| in training data:', np.abs(d['dcoords']).max())

# Multi-seed comparison
Trains 5 baseline and 5 HNN models from scratch (seeds 0 to 4) on the same data, then compares them on a fixed test set: derivative error, then energy drift over rollouts. Models are saved to `experiment-3body/3D/seed_runs/`, separate from the models used above.

In [ ]:
# Trains 10 models (about 15 to 20 minutes). Skip this cell if seed_runs/ already
# holds the models you want; the next cell only reads them.
import sys
!{sys.executable} -u experiment-3body/3D/run_seeds.py --test_trials 2000

In [ ]:
# Rolls out the saved models on 500 test orbits and measures energy drift (under a minute)
import sys
from IPython.display import Image, display
!{sys.executable} -u experiment-3body/3D/energy_drift.py
display(Image('experiment-3body/3D/seed_runs/energy_drift.png'))  

# 2D



In [ ]:
# Creates dataset for model training and visualization
!{sys.executable} experiment-3body/2D/make_dataset.py


In [ ]:
# Prints dataset and gives info regarding each feature
!{sys.executable} experiment-3body/2D/show_dataset.py


In [ ]:
!{sys.executable} experiment-3body/2D/train.py --verbose


In [ ]:
%run experiment-3body/2D/visuals.py


In [ ]:
# Loads baseline and hnn models (models need to be trained first before visualizations related to models can be displayed)

def load_model(args, baseline=False):
    output_dim = args.input_dim if baseline else 2
    nn_model = MLP(args.input_dim, args.hidden_dim, output_dim, args.nonlinearity)
    model = HNN(args.input_dim, differentiable_model=nn_model,
            field_type=args.field_type, baseline=baseline)
    case = 'baseline' if baseline else 'hnn'
    path = "{}/{}-orbits-{}.tar".format(args.save_dir, args.name, case)
    print(path)
    model.load_state_dict(torch.load(path))
    return model

args = ObjectView(get_args())
base_model = load_model(args, baseline=True)
hnn_model = load_model(args, baseline=False)


In [ ]:
plot_ground_truth()


In [ ]:
what_has_baseline_learned(base_model, args)


In [ ]:
what_has_hnn_learned(hnn_model, args)
